# Simuladores: El modelo de Bush y Mosteller
### Capítulo 10 — *Aprendizaje y Comportamiento Adaptable: Principios y Modelos*
**Arturo Bouzas** · Facultad de Psicología, UNAM · bouzaslab25.com

---
Ejecutar las celdas en orden.


In [ ]:
#@title **Simulador 10.1** — El Integrador con Fuga
# ============================================================
# Simulador 10.1 — El Integrador con Fuga
# Capítulo 10: El Modelo de Bush y Mosteller
# Aprendizaje y Comportamiento Adaptable: Principios y Modelos
# Arturo Bouzas
# ============================================================

# IMPORTACIONES
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings("ignore")

# ── Paletas claro / oscuro ────────────────────────────────────
_PALETAS_81 = {
    'claro': dict(
        azul       = '#2C5282',
        naranja    = '#C05621',
        verde      = '#276749',
        gris       = '#718096',
        fig_bg     = 'white',
        ax_bg      = 'white',
        texto      = '#2D3748',
        legend_bg  = 'white',
        panel_bg   = '#EBF4FF',
        panel_bord = '#2C5282',
        header_bg  = '#2C5282',
        header_fg  = 'white',
        sec_color  = '#2C5282',
    ),
    'oscuro': dict(
        azul       = '#90CDF4',
        naranja    = '#FBD38D',
        verde      = '#9AE6B4',
        gris       = '#A0AEC0',
        fig_bg     = '#1A202C',
        ax_bg      = '#2D3748',
        texto      = '#E2E8F0',
        legend_bg  = '#2D3748',
        panel_bg   = '#2D3748',
        panel_bord = '#4A5568',
        header_bg  = '#1A365D',
        header_fg  = '#EBF8FF',
        sec_color  = '#90CDF4',
    ),
}

def _p_81(oscuro: bool) -> dict:
    return _PALETAS_81['oscuro'] if oscuro else _PALETAS_81['claro']


# ── Lógica del modelo ─────────────────────────────────────────
def simular_81(alpha, V0, n_adq, n_ext, prob, seed=None):
    if seed is not None:
        np.random.seed(int(seed))

    total = n_adq + n_ext
    V = float(V0)
    Vs, deltas, Rs = [], [], []

    for t in range(total):
        es_ext = t >= n_adq
        R      = 0.0 if es_ext else (1.0 if np.random.random() < prob else 0.0)
        delta  = R - V
        V     += alpha * delta
        Vs.append(V)
        deltas.append(delta)
        Rs.append(R)

    return np.array(Vs), np.array(deltas), np.array(Rs)


# ── Visualización ────────────────────────────────────────────
def graficar_81(alpha, alpha2_on, alpha2,
             V0, n_adq, n_ext, prob, seed,
             mostrar_tabla, tema):

    oscuro = (tema == 'Oscuro')
    p = _p_81(oscuro)

    plt.rcParams.update({
        'font.family'       : 'serif',
        'figure.facecolor'  : p['fig_bg'],
        'axes.facecolor'    : p['ax_bg'],
        'axes.edgecolor'    : p['gris'],
        'axes.spines.top'   : False,
        'axes.spines.right' : False,
        'axes.grid'         : True,
        'grid.alpha'        : 0.30,
        'grid.color'        : p['gris'],
        'axes.labelcolor'   : p['gris'],
        'xtick.color'       : p['gris'],
        'ytick.color'       : p['gris'],
        'text.color'        : p['texto'],
        'legend.facecolor'  : p['legend_bg'],
        'legend.edgecolor'  : p['gris'],
        'legend.labelcolor' : p['texto'],
        'axes.labelsize'    : 11,
        'xtick.labelsize'   : 10,
        'ytick.labelsize'   : 10,
    })

    Vs, deltas, Rs = simular_81(alpha, V0, n_adq, n_ext, prob, seed)
    total    = n_adq + n_ext
    ensayos  = np.arange(1, total + 1)

    if alpha2_on:
        Vs2, deltas2, _ = simular_81(alpha2, V0, n_adq, n_ext, prob, seed)

    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(9, 6.5), sharex=True,
        gridspec_kw={'height_ratios': [3, 1.6], 'hspace': 0.06}
    )
    fig.patch.set_facecolor(p['fig_bg'])
    for ax in (ax1, ax2):
        ax.set_facecolor(p['ax_bg'])

    mk = 'o' if total <= 25 else None
    ax1.plot(ensayos, Vs,
             color=p['azul'], linewidth=1.5,
             marker=mk, markersize=4, markerfacecolor=p['azul'],
             label=f'V   (α = {alpha:.2f})')

    if alpha2_on:
        ax1.plot(ensayos, Vs2,
                 color=p['naranja'], linewidth=1.5,
                 marker=mk, markersize=4, markerfacecolor=p['naranja'],
                 linestyle='--', label=f'V   (α = {alpha2:.2f})')

    ax1.axhline(prob, color=p['verde'], linewidth=1.2,
                linestyle='--', alpha=0.85,
                label=f'equilibrio teórico = {prob:.2f}')
    ax1.set_ylim(-0.05, 1.18)
    ax1.set_ylabel('Valor predictivo  V')
    ax1.legend(fontsize=9, loc='lower right', framealpha=0.9)

    ax2.plot(ensayos, deltas,
             color=p['naranja'], linewidth=1.8,
             linestyle='--', label=f'δ = R − V   (α = {alpha:.2f})')

    if alpha2_on:
        ax2.plot(ensayos, deltas2,
                 color=p['azul'], linewidth=1.8,
                 linestyle=':', label=f'δ = R − V   (α = {alpha2:.2f})')

    ax2.axhline(0, color=p['gris'], linewidth=0.8, alpha=0.6)
    ax2.set_ylim(-1.15, 1.15)
    ax2.set_ylabel('Error de predicción  δ')
    ax2.set_xlabel('Ensayo')
    ax2.legend(fontsize=9, loc='upper right', framealpha=0.9)

    if n_ext > 0:
        xdiv = n_adq + 0.5
        for ax in (ax1, ax2):
            ax.axvline(xdiv, color=p['gris'], linewidth=1.2,
                       linestyle=':', alpha=0.9)
        ax1.text(xdiv + 0.15, 1.10, 'extinción →',
                 color=p['gris'], fontsize=9, va='top')

    V_adq  = Vs[n_adq - 1]
    titulo = (f'α = {alpha:.2f}   |   V₀ = {V0:.1f}   |   '
              f'P(refuerzo) = {prob:.1f}   |   '
              f'V final adq. = {V_adq:.3f}')
    if n_ext > 0:
        titulo += f'   |   V final ext. = {Vs[-1]:.3f}'
    fig.suptitle(titulo, fontsize=9.5, color=p['gris'], y=0.99, fontweight = "bold")

    plt.tight_layout()
    plt.show()

    md_adq = (
        rf"**Ecuación — Adquisición:**"
        rf"$$V_{{t+1}} = (1-{alpha:.2f})\cdot V_t + {alpha:.2f}\cdot R_t"
        rf"\qquad\Longleftrightarrow\qquad"
        rf"\Delta V = {alpha:.2f}\,(R_t - V_t)$$"
    )
    display(Markdown(md_adq))

    if n_ext > 0:
        md_ext = (
            rf"**Ecuación — Extinción** $(R_t = 0)$:"
            rf"$$V_{{t+1}} = (1-{alpha:.2f})\cdot V_t"
            rf"\qquad\Longleftrightarrow\qquad"
            rf"\Delta V = {alpha:.2f}\,(0 - V_t) = -{alpha:.2f}\,V_t$$"
        )
        display(Markdown(md_ext))

    if mostrar_tabla and total <= 20:
        _mostrar_tabla_numerica_81(alpha, V0, n_adq, n_ext, prob, seed)


def _mostrar_tabla_numerica_81(alpha, V0, n_adq, n_ext, prob, seed):
    if seed is not None:
        np.random.seed(int(seed))

    total = n_adq + n_ext
    V = float(V0)

    filas = [
        "| Ensayo | Fase | $R_t$ | $V_t$ | $\\delta_t = R_t - V_t$ | $V_{t+1}$ |",
        "|:------:|:----:|:-----:|:-----:|:----------------------:|:---------:|",
    ]
    for t in range(total):
        fase  = "Adquisición" if t < n_adq else "Extinción"
        R     = 0.0 if t >= n_adq else (1.0 if np.random.random() < prob else 0.0)
        Vt    = V
        delta = R - V
        Vt1   = V + alpha * delta
        filas.append(
            f"| {t+1} | {fase} | {R:.0f} | {Vt:.4f} | {delta:+.4f} | {Vt1:.4f} |"
        )
        V = Vt1

    display(Markdown(
        "\n**Tabla numérica — trayectoria ensayo a ensayo:**\n\n"
        + "\n".join(filas)
    ))


def _html_header_81(oscuro: bool) -> str:
    p = _p_81(oscuro)
    return (
        f'<div style="'
        f'background-color:{p["header_bg"]};'
        f'color:{p["header_fg"]};'
        f'font-family:Georgia,serif;'
        f'font-size:14px;font-weight:bold;'
        f'padding:8px 14px;'
        f'border-radius:6px 6px 0 0;'
        f'letter-spacing:0.5px;">'
        f'&nbsp;Simulador 10.1 &mdash; El Integrador con Fuga'
        f'</div>'
    )

def _html_sec_81(texto: str, oscuro: bool) -> str:
    p = _p_81(oscuro)
    return (
        f'<div style="'
        f'color:{p["sec_color"]};'
        f'font-family:Georgia,serif;'
        f'font-size:11px;font-weight:bold;'
        f'text-transform:uppercase;letter-spacing:1px;'
        f'margin:8px 0 2px 4px;">{texto}</div>'
    )

# ── Controles ─────────────────────────────────────────────────
estilo_81   = {'description_width': '130px'}
layout_s_81 = widgets.Layout(width='420px')
layout_l_81 = widgets.Layout(width='500px')

w_tema_81 = widgets.ToggleButtons(
    options=['Claro', 'Oscuro'],
    value='Claro',
    description='',
    style={'button_width': '120px'},
    layout=widgets.Layout(width='auto'),
)

w_alpha_81 = widgets.FloatSlider(
    value=0.30, min=0.05, max=0.95, step=0.05,
    description='α (tasa de aprendizaje):', style=estilo_81, layout=layout_l_81,
    readout_format='.2f')

w_alpha2_on_81 = widgets.Checkbox(
    value=False, description='Comparar segundo α',
    style=estilo_81, layout=layout_l_81)

w_alpha2_81 = widgets.FloatSlider(
    value=0.10, min=0.05, max=0.95, step=0.05,
    description='α₂ (comparación):', style=estilo_81, layout=layout_l_81,
    readout_format='.2f')

w_V0_81 = widgets.FloatSlider(
    value=0.0, min=0.0, max=1.0, step=0.1,
    description='V₀ (valor inicial):', style=estilo_81, layout=layout_s_81,
    readout_format='.1f')

w_nadq_81 = widgets.IntSlider(
    value=10, min=2, max=25, step=1,
    description='Ensayos adquisición:', style=estilo_81, layout=layout_s_81)

w_next_81 = widgets.IntSlider(
    value=8, min=0, max=25, step=1,
    description='Ensayos extinción:', style=estilo_81, layout=layout_s_81)

w_prob_81 = widgets.FloatSlider(
    value=1.0, min=0.1, max=1.0, step=0.1,
    description='P(refuerzo):', style=estilo_81, layout=layout_l_81,
    readout_format='.1f')

w_tabla_81 = widgets.Checkbox(
    value=False, description='Mostrar tabla numérica (n ensayos ≤ 20)',
    style=estilo_81, layout=layout_l_81)

btn_ejemplo_81 = widgets.Button(
    description='▶  Ejemplo del capítulo (α=0.5, 4+3)',
    button_style='',
    layout=widgets.Layout(width='340px', height='32px'),
)

def _cargar_ejemplo_81(_):
    w_alpha_81.value     = 0.50
    w_V0_81.value        = 0.0
    w_nadq_81.value      = 4
    w_next_81.value      = 3
    w_prob_81.value      = 1.0
    w_alpha2_on_81.value = False
    w_tabla_81.value     = True

btn_ejemplo_81.on_click(_cargar_ejemplo_81)

w_header_81 = widgets.HTML(value=_html_header_81(False))
w_sec1_81   = widgets.HTML(value=_html_sec_81('Tema', False))
w_sec2_81   = widgets.HTML(value=_html_sec_81('Parámetros principales', False))
w_sec3_81   = widgets.HTML(value=_html_sec_81('Comparación de α', False))
w_sec5_81   = widgets.HTML(value=_html_sec_81('Opciones de visualización', False))

_body_layout_81 = widgets.Layout(
    padding='10px 16px 14px 16px',
    background_color=_PALETAS_81['claro']['panel_bg'],
    border=f'1px solid {_PALETAS_81["claro"]["panel_bord"]}',
    border_radius='0 0 6px 6px',
)

_body_81 = widgets.VBox(
    [
        w_sec1_81, w_tema_81,
        w_sec2_81,
        w_alpha_81,
        widgets.HBox([w_V0_81]),
        widgets.HBox([w_nadq_81, w_next_81]),
        w_prob_81,
        w_sec3_81,
        w_alpha2_on_81,
        w_alpha2_81,
        w_sec5_81,
        w_tabla_81,
        widgets.HBox([btn_ejemplo_81]),
    ],
    layout=_body_layout_81,
)

ui_81 = widgets.VBox([w_header_81, _body_81])

def _actualizar_tema_81(change):
    oscuro = (change['new'] == 'Oscuro')
    p      = _p_81(oscuro)

    w_header_81.value = _html_header_81(oscuro)
    w_sec1_81.value   = _html_sec_81('Tema', oscuro)
    w_sec2_81.value   = _html_sec_81('Parámetros principales', oscuro)
    w_sec3_81.value   = _html_sec_81('Comparación de α', oscuro)
    w_sec5_81.value   = _html_sec_81('Opciones de visualización', oscuro)

    _body_81.layout.background_color = p['panel_bg']
    _body_81.layout.border           = f'1px solid {p["panel_bord"]}'

w_tema_81.observe(_actualizar_tema_81, names='value')

out_81 = widgets.interactive_output(
    graficar_81,
    {
        'alpha'        : w_alpha_81,
        'alpha2_on'    : w_alpha2_on_81,
        'alpha2'       : w_alpha2_81,
        'V0'           : w_V0_81,
        'n_adq'        : w_nadq_81,
        'n_ext'        : w_next_81,
        'prob'         : w_prob_81,
        'seed'         : widgets.fixed(42),
        'mostrar_tabla': w_tabla_81,
        'tema'         : w_tema_81,
    }
)

display(ui_81, out_81)

Output()

In [ ]:
#@title **Simulador 10.2** — Las Tres Interpretaciones del Modelo de B&M
# ============================================================
# Simulador 10.2 — Las Tres Interpretaciones del Modelo de B&M
# Capítulo 10: El Modelo de Bush y Mosteller
# Aprendizaje y Comportamiento Adaptable: Principios y Modelos
# Arturo Bouzas
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML
import warnings
warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────────────────────
# Paletas de color — tema claro y oscuro
# ──────────────────────────────────────────────────────────────

_PALETAS_82 = {
    'claro': dict(
        azul       = '#2C5282',
        naranja    = '#C05621',
        verde      = '#276749',
        gris       = '#718096',
        fig_bg     = 'white',
        ax_bg      = 'white',
        texto      = '#2D3748',
        legend_bg  = 'white',
        panel_bg   = '#EBF4FF',
        panel_bord = '#2C5282',
        header_bg  = '#2C5282',
        header_fg  = 'white',
        sec_color  = '#2C5282',
    ),
    'oscuro': dict(
        azul       = '#90CDF4',
        naranja    = '#FBD38D',
        verde      = '#9AE6B4',
        gris       = '#A0AEC0',
        fig_bg     = '#1A202C',
        ax_bg      = '#2D3748',
        texto      = '#E2E8F0',
        legend_bg  = '#2D3748',
        panel_bg   = '#2D3748',
        panel_bord = '#4A5568',
        header_bg  = '#1A365D',
        header_fg  = '#EBF8FF',
        sec_color  = '#90CDF4',
    ),
}

def _p_82(oscuro: bool) -> dict:
    return _PALETAS_82['oscuro'] if oscuro else _PALETAS_82['claro']

def _rc_82(t: dict) -> dict:
    return {
        'font.family':        'serif',
        'figure.facecolor':   t['fig_bg'],
        'axes.facecolor':     t['ax_bg'],
        'axes.spines.top':    False,
        'axes.spines.right':  False,
        'axes.edgecolor':     t['panel_bord'],
        'axes.labelcolor':    t['gris'],
        'xtick.color':        t['gris'],
        'ytick.color':        t['gris'],
        'text.color':         t['texto'],
        'axes.titlecolor':    t['azul'],
        'legend.facecolor':   t['legend_bg'],
        'legend.edgecolor':   t['panel_bord'],
        'axes.grid':          True,
        'grid.alpha':         0.35,
        'grid.color':         t['gris'],
        'axes.labelsize':     10,
        'xtick.labelsize':    9,
        'ytick.labelsize':    9,
    }

def _html_header_82(oscuro: bool) -> str:
    p = _p_82(oscuro)
    return (
        f'<div style="'
        f'background-color:{p["header_bg"]};'
        f'color:{p["header_fg"]};'
        f'font-family:Georgia,serif;'
        f'font-size:14px;font-weight:bold;'
        f'padding:8px 14px;'
        f'border-radius:6px 6px 0 0;'
        f'letter-spacing:0.5px;">'
        f'&nbsp;Simulador 10.2 &mdash; Las Tres Interpretaciones del Modelo de B&M'
        f'</div>'
    )

def _html_sec_82(texto: str, oscuro: bool) -> str:
    p = _p_82(oscuro)
    return (
        f'<div style="'
        f'color:{p["sec_color"]};'
        f'font-family:Georgia,serif;'
        f'font-size:11px;font-weight:bold;'
        f'text-transform:uppercase;letter-spacing:1px;'
        f'margin:8px 0 2px 4px;">{texto}</div>'
    )

def _tabla_html_82(alpha: float, k_max: int, t: dict) -> str:
    ks    = np.arange(k_max)
    pesos = alpha * (1 - alpha) ** ks
    acum  = np.cumsum(pesos)
    mitad = int(np.ceil(np.log(0.5) / np.log(max(1 - alpha, 1e-9))))

    if alpha <= 0.15:
        etiqueta = 'Memoria larga  (entorno estable)'
    elif alpha >= 0.55:
        etiqueta = 'Memoria corta  (entorno cambiante)'
    else:
        etiqueta = 'Memoria intermedia'

    filas = ''
    for k in range(k_max):
        bg    = t['panel_bg'] if k % 2 == 1 else t['fig_bg']
        negri = ' font-weight:bold;' if k == 0 else ''
        barra = int(pesos[k] / pesos[0] * 60)
        bar_s = f'<span style="display:inline-block;width:{barra}px;height:9px;' \
                f'background:{t["azul"]};opacity:0.55;vertical-align:middle;' \
                f'border-radius:2px;"></span>'
        filas += (
            f'<tr style="background:{bg}; color:{t["texto"]};{negri}">'
            f'<td style="text-align:center;padding:5px 14px;">{k}</td>'
            f'<td style="text-align:right;padding:5px 14px;font-family:monospace;">'
            f'{pesos[k]:.5f}</td>'
            f'<td style="text-align:right;padding:5px 14px;">{pesos[k]*100:.2f}%</td>'
            f'<td style="text-align:right;padding:5px 14px;">{acum[k]:.5f}</td>'
            f'<td style="text-align:right;padding:5px 14px;">{acum[k]*100:.1f}%</td>'
            f'<td style="padding:5px 14px;">{bar_s}</td>'
            f'</tr>'
        )

    suma_total = pesos.sum()
    nota_suma  = (
        f'Suma de los {k_max} pesos mostrados: <b>{suma_total:.5f}</b>'
        + (f'  (la suma infinita converge a 1.000)' if suma_total < 0.999 else '  ≈ 1.000')
    )

    html = f"""
    <div style="margin-top:20px; font-family:serif;">
      <h4 style="color:{t['azul']}; margin-bottom:6px; font-size:14px;">
        ▸ Pesos del filtro de media exponencial corrida — α = {alpha:.2f}
      </h4>
      <p style="color:{t['gris']}; font-size:12px; margin:0 0 10px 0;">
        {etiqueta} · mitad del peso acumulado en los primeros
        <b>{mitad}</b> ensayo{'s' if mitad != 1 else ''} atrás
      </p>

      <table style="border-collapse:collapse; font-size:13px; min-width:520px;">
        <thead>
          <tr style="background:{t['header_bg']}; color:{t['header_fg']};">
            <th style="padding:8px 14px; text-align:center; border-bottom:2px solid {t['panel_bord']};">k (ensayos atrás)</th>
            <th style="padding:8px 14px; text-align:right; border-bottom:2px solid {t['panel_bord']};">Peso α(1−α)ᵏ</th>
            <th style="padding:8px 14px; text-align:right; border-bottom:2px solid {t['panel_bord']};">Porcentaje</th>
            <th style="padding:8px 14px; text-align:right; border-bottom:2px solid {t['panel_bord']};">Peso acumulado</th>
            <th style="padding:8px 14px; text-align:right; border-bottom:2px solid {t['panel_bord']};">% acumulado</th>
            <th style="padding:8px 14px; border-bottom:2px solid {t['panel_bord']};">Proporción relativa</th>
          </tr>
        </thead>
        <tbody>{filas}</tbody>
      </table>

      <p style="color:{t['gris']}; font-size:11.5px; margin-top:9px;">{nota_suma}</p>
      <p style="color:{t['gris']}; font-size:11.5px; margin-top:3px;">
        El peso del ensayo más reciente (k = 0) es exactamente α = {alpha:.2f}.
        Cada ensayo adicional hacia el pasado reduce el peso por el factor (1−α) = {1-alpha:.2f}.
      </p>
    </div>
    """
    return html


def graficar_82(alpha=0.30, n_adq=15, n_ext=0, v0=0.0, prob=1.0, semilla=42, tema='Claro'):
    oscuro = (tema == 'Oscuro')
    t = _p_82(oscuro)
    plt.rcParams.update(_rc_82(t))

    rng = np.random.default_rng(int(semilla))
    V      = float(v0)
    Vs     = [V]
    deltas = []
    Rs     = []

    for _ in range(n_adq):
        R     = 1 if rng.random() < prob else 0
        delta = R - V
        V    += alpha * delta
        Vs.append(V); deltas.append(delta); Rs.append(R)

    for _ in range(n_ext):
        delta = 0.0 - V
        V    += alpha * delta
        Vs.append(V); deltas.append(delta); Rs.append(0)

    n_total = n_adq + n_ext
    ens_adq = np.arange(1, n_adq + 1)
    Vs_adq  = Vs[1:n_adq + 1]
    del_adq = deltas[:n_adq]

    if n_ext > 0:
        ens_ext = np.arange(n_adq + 1, n_total + 1)
        Vs_ext  = Vs[n_adq + 1:]
        del_ext = deltas[n_adq:]
    else:
        ens_ext = Vs_ext = del_ext = []

    k_max = min(20, max(n_adq, 5))
    ks    = np.arange(k_max)
    pesos = alpha * (1 - alpha) ** ks

    fig, axes = plt.subplots(1, 3, figsize=(20, 6.5))
    fig.patch.set_facecolor(t['fig_bg'])

    par_str = (
        f'α = {alpha:.2f}  |  P(R) = {prob:.1f}  |  '
        f'V₀ = {v0:.2f}  |  adquisición: {n_adq} ensayos'
        + (f'  |  extinción: {n_ext} ensayos' if n_ext > 0 else '')
    )
    fig.suptitle(par_str, fontsize=9.5, color=t['gris'], fontweight="bold", y=1.005)

    def _fase_sep(ax):
        if n_ext > 0:
            ax.axvline(n_adq + 0.5, color=t['gris'], lw=0.9, linestyle='--', alpha=0.55, zorder=1)
            ymax, ymin = ax.get_ylim()
            rng_y = ymax - ymin
            ax.text(n_adq * 0.5, ymax - rng_y * 0.06, 'Adquisición', ha='center', fontsize=7.5, color=t['azul'], alpha=0.8)
            ax.text(n_adq + n_ext * 0.5 + 0.5, ymax - rng_y * 0.06, 'Extinción', ha='center', fontsize=7.5, color=t['naranja'], alpha=0.8)

    # Panel 1: Integrador
    ax1 = axes[0]
    mk, ms = ('o', 4) if n_total <= 30 else (None, 0)
    ax1.plot(ens_adq, Vs_adq, color=t['azul'], lw=2.3, marker=mk, ms=ms, label='Adquisición', zorder=3)
    if n_ext > 0:
        ax1.plot([n_adq, n_adq + 1], [Vs_adq[-1], Vs_ext[0]], color=t['gris'], lw=1.2, ls=':', zorder=2)
        ax1.plot(ens_ext, Vs_ext, color=t['naranja'], lw=2.3, marker=mk, ms=ms, label='Extinción', zorder=3)
    ax1.axhline(prob, color=t['verde'], lw=1.3, ls='--', alpha=0.85, label=f'equilibrio = {prob:.2f}')
    if v0 > 0.02:
        ax1.axhline(v0, color=t['gris'], lw=0.9, ls=':', alpha=0.5)
        ax1.text(0.5, v0 + 0.02, f'V₀ = {v0:.2f}', fontsize=7.5, color=t['gris'], ha='left')
    ax1.set_ylim(-0.05, 1.22)
    ax1.set_xlim(0.5, n_total + 0.5)
    ax1.set_xlabel('Ensayo', color=t['gris'])
    ax1.set_ylabel('V  (valor predictivo)', color=t['gris'])
    ax1.set_title('Panel 1\nIntegrador con fuga', fontsize=10, color=t['azul'], pad=8, fontweight='semibold')
    ax1.legend(fontsize=8, framealpha=0.8, loc='upper left')
    _fase_sep(ax1)
    ax1.text(0.5, -0.35, r'$V_{t+1} = (1-\alpha)\,V_t + \alpha\,R_t$', transform=ax1.transAxes, ha='center', fontsize=9, color=t['azul'], bbox=dict(facecolor=t['ax_bg'], edgecolor=t['azul'], alpha=0.85, boxstyle='round,pad=0.35'))

    # Panel 2: Error δ
    ax2 = axes[1]
    ax2.plot(ens_adq, del_adq, color=t['naranja'], lw=2.0, ls='--', alpha=0.9, label='δ adq.', zorder=3)
    ax2.fill_between(ens_adq, del_adq, 0, where=np.array(del_adq) > 0, color=t['naranja'], alpha=0.20)
    ax2.fill_between(ens_adq, del_adq, 0, where=np.array(del_adq) < 0, color=t['naranja'], alpha=0.20)
    if n_ext > 0:
        ax2.plot(ens_ext, del_ext, color=t['sec_color'], lw=2.0, ls='--', alpha=0.9, label='δ ext.', zorder=3)
        ax2.fill_between(ens_ext, del_ext, 0, color=t['sec_color'], alpha=0.15)
    ax2.axhline(0, color=t['gris'], lw=0.9, alpha=0.65)
    ax2.set_ylim(-1.15, 1.15)
    ax2.set_xlim(0.5, n_total + 0.5)
    ax2.set_xlabel('Ensayo', color=t['gris'])
    ax2.set_ylabel('δ = R − V', color=t['gris'])
    ax2.set_title('Panel 2\nError de predicción δ', fontsize=10, color=t['azul'], pad=8, fontweight='semibold')
    ax2.legend(fontsize=7.5, framealpha=0.8, loc='upper right')
    _fase_sep(ax2)
    ax2.text(0.5, -0.35, r'$\Delta V = \alpha\,\delta_t$', transform=ax2.transAxes, ha='center', fontsize=9, color=t['naranja'], bbox=dict(facecolor=t['ax_bg'], edgecolor=t['naranja'], alpha=0.85, boxstyle='round,pad=0.35'))

    # Panel 3: Filtro Exponencial
    ax3 = axes[2]
    bars = ax3.bar(ks, pesos, color=t['azul'], alpha=0.68, width=0.68, edgecolor=t['azul'], lw=0.8, zorder=3)
    bars[0].set_alpha(0.95); bars[0].set_edgecolor(t['azul']); bars[0].set_linewidth(1.8)
    ax3.set_xlabel('k  (ensayos atrás)', color=t['gris'])
    ax3.set_ylabel(r'Peso  α · (1 − α)$^k$', color=t['gris'])
    ax3.set_title('Panel 3\nFiltro de media exponencial', fontsize=10, color=t['azul'], pad=8, fontweight='semibold')
    ax3.annotate(f'k=0: {alpha*100:.0f}%', xy=(0, pesos[0]), xytext=(min(3, k_max - 1), pesos[0] * 0.8), fontsize=8, color=t['azul'], arrowprops=dict(arrowstyle='->', color=t['azul'], lw=1.2))
    ax3.text(0.5, -0.35, r'$V_t = \sum_{k=0}^{\infty} \alpha\,(1-\alpha)^k\,R_{t-k}$', transform=ax3.transAxes, ha='center', fontsize=9, color=t['verde'], bbox=dict(facecolor=t['ax_bg'], edgecolor=t['verde'], alpha=0.85, boxstyle='round,pad=0.35'))

    plt.tight_layout(rect=[0, 0.10, 1, 1])
    plt.show()

    display(HTML(_tabla_html_82(alpha, k_max, t)))

estilo_82   = {'description_width': '140px'}
layout_s_82 = widgets.Layout(width='420px')
layout_l_82 = widgets.Layout(width='500px')

w_tema_82 = widgets.ToggleButtons(
    options=['Claro', 'Oscuro'],
    value='Claro',
    description='',
    style={'button_width': '120px'},
    layout=widgets.Layout(width='auto'),
)

w_alpha_82 = widgets.FloatSlider(value=0.30, min=0.05, max=0.95, step=0.05, description='α (tasa aprendizaje):', style=estilo_82, layout=layout_l_82, readout_format='.2f')
w_v0_82    = widgets.FloatSlider(value=0.0, min=0.0, max=0.5, step=0.05, description='V₀ (valor inicial):', style=estilo_82, layout=layout_s_82, readout_format='.2f')
w_prob_82  = widgets.FloatSlider(value=1.0, min=0.1, max=1.0, step=0.1, description='P(refuerzo):', style=estilo_82, layout=layout_s_82, readout_format='.1f')
w_nadq_82  = widgets.IntSlider(value=15, min=5, max=40, step=1, description='Ensayos adq.:', style=estilo_82, layout=layout_s_82)
w_next_82  = widgets.IntSlider(value=0, min=0, max=20, step=1, description='Ensayos ext.:', style=estilo_82, layout=layout_s_82)

w_header_82 = widgets.HTML(value=_html_header_82(False))
w_sec1_82   = widgets.HTML(value=_html_sec_82('Tema', False))
w_sec2_82   = widgets.HTML(value=_html_sec_82('Parámetros de Simulación', False))

_body_layout_82 = widgets.Layout(
    padding='10px 16px 14px 16px',
    background_color=_PALETAS_82['claro']['panel_bg'],
    border=f'1px solid {_PALETAS_82["claro"]["panel_bord"]}',
    border_radius='0 0 6px 6px',
)

_body_82 = widgets.VBox(
    [
        w_sec1_82, w_tema_82,
        w_sec2_82,
        w_alpha_82,
        widgets.HBox([w_v0_82, w_prob_82]),
        widgets.HBox([w_nadq_82, w_next_82]),
    ],
    layout=_body_layout_82,
)

ui_82 = widgets.VBox([w_header_82, _body_82])

def _actualizar_tema_82(change):
    oscuro = (change['new'] == 'Oscuro')
    p      = _p_82(oscuro)

    w_header_82.value = _html_header_82(oscuro)
    w_sec1_82.value   = _html_sec_82('Tema', oscuro)
    w_sec2_82.value   = _html_sec_82('Parámetros de Simulación', oscuro)

    _body_82.layout.background_color = p['panel_bg']
    _body_82.layout.border           = f'1px solid {p["panel_bord"]}'

w_tema_82.observe(_actualizar_tema_82, names='value')

out_82 = widgets.interactive_output(
    graficar_82,
    {
        'alpha':   w_alpha_82,
        'n_adq':   w_nadq_82,
        'n_ext':   w_next_82,
        'v0':      w_v0_82,
        'prob':    w_prob_82,
        'semilla': widgets.fixed(42),
        'tema':    w_tema_82,
    }
)

display(ui_82, out_82)

Output()

In [ ]:
#@title **Simulador 10.3** — Media y Mediana Corrida
# ============================================================
# Simulador 10.3 — Media y Mediana Corrida
# Capítulo 10: El Modelo de Bush y Mosteller
# Aprendizaje y Comportamiento Adaptable: Principios y Modelos
# Arturo Bouzas
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML
import warnings
warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────────────────────
# Paletas de color — tema claro y oscuro
# ──────────────────────────────────────────────────────────────
_PALETAS_83 = {
    'claro': dict(
        azul       = '#2C5282',
        naranja    = '#C05621',
        verde      = '#276749',
        gris       = '#718096',
        fig_bg     = 'white',
        ax_bg      = 'white',
        texto      = '#2D3748',
        legend_bg  = 'white',
        panel_bg   = '#EBF4FF',
        panel_bord = '#2C5282',
        header_bg  = '#2C5282',
        header_fg  = 'white',
        sec_color  = '#2C5282',
    ),
    'oscuro': dict(
        azul       = '#90CDF4',
        naranja    = '#FBD38D',
        verde      = '#9AE6B4',
        gris       = '#A0AEC0',
        fig_bg     = '#1A202C',
        ax_bg      = '#2D3748',
        texto      = '#E2E8F0',
        legend_bg  = '#2D3748',
        panel_bg   = '#2D3748',
        panel_bord = '#4A5568',
        header_bg  = '#1A365D',
        header_fg  = '#EBF8FF',
        sec_color  = '#90CDF4',
    ),
}

def _p_83(oscuro: bool) -> dict:
    return _PALETAS_83['oscuro'] if oscuro else _PALETAS_83['claro']

def _rc_83(t: dict) -> dict:
    return {
        'font.family':        'serif',
        'figure.facecolor':   t['fig_bg'],
        'axes.facecolor':     t['ax_bg'],
        'axes.spines.top':    False,
        'axes.spines.right':  False,
        'axes.edgecolor':     t['panel_bord'],
        'axes.labelcolor':    t['gris'],
        'xtick.color':        t['gris'],
        'ytick.color':        t['gris'],
        'text.color':         t['texto'],
        'axes.titlecolor':    t['azul'],
        'legend.facecolor':   t['legend_bg'],
        'legend.edgecolor':   t['panel_bord'],
        'axes.grid':          True,
        'grid.alpha':         0.35,
        'grid.color':         t['gris'],
        'axes.labelsize':     10,
        'xtick.labelsize':    9,
        'ytick.labelsize':    9,
    }

# ──────────────────────────────────────────────────────────────
# DATOS DEL CAPÍTULO 8 (A y B)
# ──────────────────────────────────────────────────────────────
# PANEL A: Datos originales con valor aberrante
DATOS_A = np.array([10, 14, 11, 16, 13, 9, 85, 12, 18, 15, 20, 17, 13, 19, 16],
                   dtype=float)
IDX_ABERRANTE_A = 7

# PANEL B: Datos nuevos con mayor varianza
DATOS_B = np.array([12, 15, 14, 18, 7, 5, 22, 20, 24, 19, 8, 6, 30, 28, 32, 27, 10, 9, 35, 38, 40],
                   dtype=float)
IDX_ABERRANTE_B = None

# ──────────────────────────────────────────────────────────────
# FUNCIONES HTML Y DE CÁLCULO
# ──────────────────────────────────────────────────────────────
def _html_header_83(oscuro: bool) -> str:
    p = _p_83(oscuro)
    return (
        f'<div style="'
        f'background-color:{p["header_bg"]};'
        f'color:{p["header_fg"]};'
        f'font-family:Georgia,serif;'
        f'font-size:14px;font-weight:bold;'
        f'padding:8px 14px;'
        f'border-radius:6px 6px 0 0;'
        f'letter-spacing:0.5px;">'
        f'&nbsp;Simulador 10.3 &mdash; Media y Mediana Corrida'
        f'</div>'
    )

def _html_sec_83(texto: str, oscuro: bool) -> str:
    p = _p_83(oscuro)
    return (
        f'<div style="'
        f'color:{p["sec_color"]};'
        f'font-family:Georgia,serif;'
        f'font-size:11px;font-weight:bold;'
        f'text-transform:uppercase;letter-spacing:1px;'
        f'margin:8px 0 2px 4px;">{texto}</div>'
    )

def _tabla_html_83(datos, ventana, idx_s, vals_med, vals_mean, t, titulo_panel, idx_aber=None):
    dias_s = set(idx_s)
    filas = ''

    for i, d in enumerate(datos):
        dia = i + 1
        bg  = t['panel_bg'] if i % 2 == 1 else t['fig_bg']

        es_aber = (idx_aber is not None and dia == idx_aber)
        negri   = ' font-weight:bold;' if es_aber else ''
        color_d = t['naranja'] if es_aber else t['texto']

        if i in dias_s:
            j       = list(idx_s).index(i)
            med_val = f'{vals_med[j]:.1f}'
            mea_val = f'{vals_mean[j]:.1f}'
            if es_aber:
                nota = f'← aberrante'
            else:
                nota = ''
        else:
            med_val = '—'
            mea_val = '—'
            nota    = '—'

        filas += (
            f'<tr style="background:{bg}; color:{t["texto"]};{negri}">'
            f'<td style="text-align:center;padding:4px 6px;color:{color_d};">{dia}</td>'
            f'<td style="text-align:right;padding:4px 6px;font-family:monospace;color:{color_d};">{int(d)}</td>'
            f'<td style="text-align:right;padding:4px 6px;color:{t["azul"]};">{med_val}</td>'
            f'<td style="text-align:right;padding:4px 6px;color:{t["naranja"]};">{mea_val}</td>'
            f'<td style="text-align:center;padding:4px 6px;font-size:10px;color:{t["gris"]};">{nota}</td>'
            f'</tr>'
        )

    nota_aberrante = f"<p style=\"color:{t['gris']}; font-size:11px; margin-top:6px;\">Fila en <b style='color:{t['naranja']};'>naranja</b> = valor aberrante.</p>" if idx_aber is not None else ""

    html = f"""
    <div style="font-family:serif;">
      <h4 style="color:{t['azul']}; margin-bottom:6px; font-size:13px;">▸ {titulo_panel}</h4>
      <table style="border-collapse:collapse; font-size:11.5px; width:100%;">
        <thead>
          <tr style="background:{t['header_bg']}; color:{t['header_fg']};">
            <th style="padding:6px; text-align:center; border-bottom:2px solid {t['panel_bord']};">Día</th>
            <th style="padding:6px; text-align:right; border-bottom:2px solid {t['panel_bord']};">Cont.</th>
            <th style="padding:6px; text-align:right; border-bottom:2px solid {t['panel_bord']};">Mediana</th>
            <th style="padding:6px; text-align:right; border-bottom:2px solid {t['panel_bord']};">Media</th>
            <th style="padding:6px; text-align:center; border-bottom:2px solid {t['panel_bord']};">Nota</th>
          </tr>
        </thead>
        <tbody>{filas}</tbody>
      </table>
      {nota_aberrante}
    </div>
    """
    return html

def _limitaciones_html_83(ventana, t):
    html = f"""
    <div style="margin-top:20px; padding-top:15px; border-top:1px solid {t['panel_bord']}; font-family:serif; font-size:13px; color:{t['texto']}; clear:both;">
      <h4 style="color:{t['azul']}; font-size:14px; margin-bottom:8px;">▸ Limitaciones del filtro de ventana {ventana} y la solución de B&M</h4>
      <table style="border-collapse:collapse; width:100%;">
        <tr><td style="padding:4px 10px; vertical-align:top; width:20px; color:{t['naranja']}; font-weight:bold;">1.</td><td style="padding:4px 10px; color:{t['texto']};">Pesos iguales dentro de la ventana.</td></tr>
        <tr><td style="padding:4px 10px; vertical-align:top; color:{t['naranja']}; font-weight:bold;">2.</td><td style="padding:4px 10px; color:{t['texto']};">Peso cero fuera de la ventana.</td></tr>
        <tr><td style="padding:4px 10px; vertical-align:top; color:{t['naranja']}; font-weight:bold;">3.</td><td style="padding:4px 10px; color:{t['texto']};">Requiere almacenar {ventana} observaciones.</td></tr>
      </table>
      <div style="margin-top:10px; padding:8px 14px; background:{t['panel_bg']}; border-left:3px solid {t['azul']}; border-radius:0 4px 4px 0;">
        <b style="color:{t['azul']};">El modelo de B&M resuelve los tres:</b> <span style="font-family:monospace; margin-left:10px;">V<sub>t+1</sub> = (1−α)·V<sub>t</sub> + α·R<sub>t</sub></span>
      </div>
    </div>
    """
    return html

def aplicar_filtro_83(datos, ventana, usar_mediana):
    mitad = ventana // 2
    n = len(datos)
    idx, vals = [], []
    for i in range(mitad, n - mitad):
        w = datos[i - mitad : i + mitad + 1]
        vals.append(np.median(w) if usar_mediana else np.mean(w))
        idx.append(i)
    return np.array(idx), np.array(vals)

def _graficar_panel(ax, datos, ventana, idx_aber, titulo, t):
    dias = np.arange(1, len(datos) + 1)
    idx_s, vals_med  = aplicar_filtro_83(datos, ventana, usar_mediana=True)
    _,     vals_mean = aplicar_filtro_83(datos, ventana, usar_mediana=False)
    dias_s = idx_s + 1
    mitad = ventana // 2

    # Datos originales
    ax.plot(dias, datos, color=t['gris'], linewidth=1.5, marker='o', markersize=4,
            markerfacecolor=t['fig_bg'], markeredgecolor=t['gris'], label='Datos', zorder=2, alpha=0.9)

    # Resaltar aberrante
    if idx_aber is not None:
        ax.plot(idx_aber, datos[idx_aber - 1], 'o', markersize=8, color=t['naranja'],
                markerfacecolor=t['naranja'], zorder=4, alpha=0.9)
        ax.annotate(f'{int(datos[idx_aber-1])}',
                    xy=(idx_aber, datos[idx_aber - 1]),
                    xytext=(idx_aber + 1, datos[idx_aber - 1] - 15),
                    fontsize=8.5, color=t['naranja'],
                    arrowprops=dict(arrowstyle='->', color=t['naranja'], lw=1), ha='left')

    ax.plot(dias_s, vals_med, color=t['azul'], linewidth=2.5, marker='s', markersize=4,
            markerfacecolor=t['azul'], label='Mediana', zorder=3)
    ax.plot(dias_s, vals_mean, color=t['naranja'], linewidth=1.8, linestyle='--', marker='^',
            markersize=4, markerfacecolor=t['naranja'], label='Media', zorder=3, alpha=0.85)

    extremos_idx  = list(range(1, mitad + 1)) + list(range(len(datos) - mitad + 1, len(datos) + 1))
    extremos_vals = [datos[i - 1] for i in extremos_idx]
    ax.scatter(extremos_idx, extremos_vals, color=t['gris'], s=15, zorder=2, alpha=0.4)

    ax.set_xlabel('Día', color=t['gris'])
    ax.set_ylabel('Contagios diarios', color=t['gris'])
    ax.set_xticks(dias[::2])
    ax.set_ylim(min(datos) - 5, max(datos) * 1.15)
    ax.legend(fontsize=8, framealpha=0.9, loc='upper left')
    ax.set_title(titulo, fontsize=10, color=t['azul'], pad=8, fontweight='bold')

    return idx_s, vals_med, vals_mean

# ──────────────────────────────────────────────────────────────
# LÓGICA PRINCIPAL DEL WIDGET
# ──────────────────────────────────────────────────────────────
def graficar_ambos_paneles(ventana=3, tema='Claro'):
    oscuro = (tema == 'Oscuro')
    t = _p_83(oscuro)
    plt.rcParams.update(_rc_83(t))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16,6))
    fig.patch.set_facecolor(t['fig_bg'])

    # Graficar Panel A
    idx_s_A, med_A, mean_A = _graficar_panel(
        ax1, DATOS_A, ventana, IDX_ABERRANTE_A,
        f'Panel A: Valor aberrante', t
    )

    # Graficar Panel B
    idx_s_B, med_B, mean_B = _graficar_panel(
        ax2, DATOS_B, ventana, IDX_ABERRANTE_B,
        f'Panel B: Alta varianza', t
    )

    plt.tight_layout(pad=2.0)
    plt.show()

    # Generar HTML de las tablas
    html_tabla_A = _tabla_html_83(DATOS_A, ventana, idx_s_A, med_A, mean_A, t, f"Tabla A", IDX_ABERRANTE_A)
    html_tabla_B = _tabla_html_83(DATOS_B, ventana, idx_s_B, med_B, mean_B, t, f"Tabla B", IDX_ABERRANTE_B)

    # Contenedor Flexbox
    html_combinado = f"""
    <div style="display: flex; flex-direction: row; justify-content: space-between; gap: 40px; width: 92%; margin-top: 10px;">
        <div style="flex: 1; min-width: 0;">
            {html_tabla_A}
        </div>
        <div style="flex: 1; min-width: 0;">
            {html_tabla_B}
        </div>
    </div>
    """

    display(HTML(html_combinado))

    # Limitaciones
    display(HTML(_limitaciones_html_83(ventana, t)))

# ──────────────────────────────────────────────────────────────
# UI Y CONTROLES
# ──────────────────────────────────────────────────────────────
estilo_83 = {'description_width': '100px'}

w_tema_83 = widgets.ToggleButtons(
    options=['Claro', 'Oscuro'],
    value='Claro',
    description='',
    style={'button_width': '120px'},
    layout=widgets.Layout(width='auto'),
)

w_vent_83 = widgets.IntSlider(
    value=3, min=3, max=9, step=2,
    description='Ventana (días):',
    style=estilo_83,
    layout=widgets.Layout(width='420px'))

w_header_83 = widgets.HTML(value=_html_header_83(False))
w_sec1_83   = widgets.HTML(value=_html_sec_83('Tema', False))
w_sec2_83   = widgets.HTML(value=_html_sec_83('Tamaño de ventana', False))

_body_layout_83 = widgets.Layout(
    padding='10px 16px 14px 16px',
    background_color=_PALETAS_83['claro']['panel_bg'],
    border=f'1px solid {_PALETAS_83["claro"]["panel_bord"]}',
    border_radius='0 0 6px 6px',
)

_body_83 = widgets.VBox(
    [w_sec1_83, w_tema_83, w_sec2_83, w_vent_83],
    layout=_body_layout_83,
)

ui_83 = widgets.VBox([w_header_83, _body_83])

def _actualizar_tema_83(change):
    oscuro = (change['new'] == 'Oscuro')
    p      = _p_83(oscuro)
    w_header_83.value = _html_header_83(oscuro)
    w_sec1_83.value   = _html_sec_83('Tema', oscuro)
    w_sec2_83.value   = _html_sec_83('Tamaño de ventana', oscuro)
    _body_83.layout.background_color = p['panel_bg']
    _body_83.layout.border           = f'1px solid {p["panel_bord"]}'

w_tema_83.observe(_actualizar_tema_83, names='value')

out_83 = widgets.interactive_output(
    graficar_ambos_paneles,
    {'ventana': w_vent_83, 'tema': w_tema_83}
)

display(ui_83, out_83)

Output()

---
## Créditos y licencia

Este notebook es parte del proyecto:

> **Bouzas, A. (2026).** *Aprendizaje y Comportamiento Adaptable: Principios y Modelos.*
> Lab25, Facultad de Psicología, UNAM.
> https://www.bouzaslab25.com

Apoyo en la construcción del simulador: **Eduardo Sánchez**.

Código disponible en: **https://github.com/bouzaslab25/libro-aca**
Licencia: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)
